In [ ]:
#Import the required modules and libraries
import pastaq as pq
import math
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as colors
from collections import defaultdict
from pathlib import Path
import re
from itertools import combinations
import ms_entropy as me
import ast
from dataclasses import dataclass
import networkx as nx



In [ ]:
cluster_plot = pd.read_excel(r"C:\Users\User\Cluster_Map.xlsx")

In [ ]:
def visualize_clusters(
    cluster_plot_excel_file,
    classes=None,
    ncols=2,
):
    """
    Plot precursor m/z versus retention time for annotated clusters.

    Parameters
    ----------
    cluster_plot_excel_file : str or pandas.DataFrame
        Excel file containing:
            - Cluster_id
            - Metabolite_Class
            - average_precursor_mz
            - average_precursor_RT

    classes : str or iterable of str, optional
        Lipid class or classes to display.
        For example: "PC" or ["PC", "PE", "PS"].
        By default, all classes are displayed.

    ncols : int, default=2
        Number of subplots per row.

    Returns
    -------
    fig, axes
        Matplotlib figure and axes.
    """

    # ----------------------------------------------------------
    # Class colours
    # ----------------------------------------------------------

    class_colors = {
        "Dodecylbenzenesulfonic acid": "azure",
        "BA": "goldenrod",
        "GPNAE": "tan",
        "SE": "bisque",
        "SL": "deeppink",
        "PC": "blue",
        "PC-O": "lightblue",
        "PE": "green",
        "PE-O": "lightgreen",
        "DMPE": "lime",
        "MMPE": "darkgreen",
        "PI": "purple",
        "PI-Cer": "rebeccapurple",
        "PS": "red",
        "PA": "brown",
        "CL": "cyan",
        "SM": "magenta",
        "GM3": "rosybrown",
        "Cer": "lavenderblush",
        "AHexCer": "palevioletred",
        "HexCer": "pink",
        "Hex2Cer": "brown",
        "Hex3Cer": "yellow",
        "SHexCer": "darkorange",
        "MGMG": "mediumaquamarine",
        "MGDG-O": "olive",
        "MGDG": "olivedrab",
        "SMGDG-O": "yellowgreen",
        "DG": "gray",
        "TG": "green",
        "TG-O": "lightgreen",
        "FA": "yellow",
        "LPA": "lightgreen",
        "LPC": "lightcoral",
        "LPC-O": "coral",
        "LPG": "lightgray",
        "LPI": "lightcyan",
        "LPS": "darkred",
        "LPE": "cadetblue",
        "PMeOH": "sienna",
        "GD2": "indianred",
        "CerP": "plum",
        "PG": "indigo",
        "NAGly": "powderblue",
        "Inosine": "wheat",
        "Unidentified": "darkgray",
        "Unclear": "gray",
        "Other": "gray",
    }

    # ----------------------------------------------------------
    # Helper: convert "571.0458 +/- 0.0021" -> 571.0458 [This is because the average_precursor_mz and average_precursor_RT columns may contain 95% CIs]
    # ----------------------------------------------------------

    def extract_mean(value):

        if value is None or pd.isna(value):
            return np.nan

        if isinstance(value, (int, float, np.integer, np.floating)):
            return float(value)

        match = re.search(
            r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)",
            str(value),
        )

        if match is None:
            return np.nan

        return float(match.group())

    # ----------------------------------------------------------
    # Load Excel file
    # ----------------------------------------------------------

    if isinstance(cluster_plot_excel_file, pd.DataFrame):
        df = cluster_plot_excel_file.copy()
    else:
        df = pd.read_excel(cluster_plot_excel_file)

    # ----------------------------------------------------------
    # Check required columns
    # ----------------------------------------------------------

    required_columns = [
        "Cluster_id",
        "Metabolite_Class",
        "average_precursor_mz",
        "average_precursor_RT",
    ]

    missing = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    # ----------------------------------------------------------
    # Filter classes
    # ----------------------------------------------------------

    if classes is not None:

        if isinstance(classes, str):
            classes = [classes]

        df = df[
            df["Metabolite_Class"].isin(classes)
        ].copy()

    # ----------------------------------------------------------
    # Extract numeric means from m/z and RT
    # ----------------------------------------------------------

    df["plot_mz"] = df["average_precursor_mz"].apply(
        extract_mean
    )

    df["plot_RT"] = df["average_precursor_RT"].apply(
        extract_mean
    )

    # Remove rows that cannot be plotted
    df = df[
        df["plot_mz"].notna()
        & df["plot_RT"].notna()
    ].copy()

    if df.empty:
        raise ValueError(
            "No valid data remain after filtering."
        )

    # ----------------------------------------------------------
    # Create one plot
    # ----------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10, 8)
    )

    # ----------------------------------------------------------
    # Plot each lipid class separately
    # ----------------------------------------------------------

    for lipid_class in df["Metabolite_Class"].unique():

        class_df = df[
            df["Metabolite_Class"] == lipid_class
        ]

        color = class_colors.get(
            lipid_class,
            "gray",
        )

        ax.scatter(
            class_df["plot_RT"],
            class_df["plot_mz"],
            s=50,
            c=color,
            label=lipid_class,
            alpha=0.8,
            edgecolors="black",
            linewidths=0.5,
        )

        # ------------------------------------------------------
        # Add cluster ID labels
        # ------------------------------------------------------

        for _, row in class_df.iterrows():

            ax.annotate(
                str(row["Cluster_id"]),
                (
                    row["plot_RT"],
                    row["plot_mz"],
                ),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )

    # ----------------------------------------------------------
    # Axis labels
    # ----------------------------------------------------------

    ax.set_xlabel(
        "Retention time"
    )

    ax.set_ylabel(
        "Precursor m/z"
    )

    ax.set_title(
        "Precursor m/z vs retention time"
    )

    ax.grid(
        True,
        alpha=0.25,
    )

    ax.legend(
        title="Metabolite class",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=4,
    )

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.20)

    return fig, ax


In [ ]:
fig, ax = visualize_clusters(
    cluster_plot
)

plt.show()